**Tracking a secondary cosmic ray muon.**
When high-energy cosmic ray protons strike the upper atmosphere they produce
**muons** - unstable particles with a rest-frame lifetime of about
$2.2\,\mu\text{s}$.
A muon travelling at close to the speed of light experiences **time dilation**:
its apparent lifetime in the lab frame is $\gamma \times 2.2\,\mu\text{s}$,
where $\gamma = 1/\sqrt{1-v^2/c^2}$.
This allows muons created 10-15 km above the surface to reach
ground-level detectors before decaying - a direct experimental confirmation
of special relativity.

---
A particle detector recorded the height of one such muon at successive times.
From the slope of the height vs. time trajectory we can extract the muon's
velocity (as a fraction of $c$) and work backwards to its origination altitude.

In [ ]:
"""cosmic_rays.ipynb"""

# Cell 01 - Load the detector data from CSV
# Column 0: time in nanoseconds (ns)
# Column 1: height above ground in centimeters (cm)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

file_name = "ray.csv"
data = np.genfromtxt(file_name, delimiter=",")
pd.DataFrame(data[:10], columns=["Time (ns)", "Height (cm)"]).style.hide(axis="index")

---
**Extracting the time and height columns.**
Column 0 is time $t$ in nanoseconds (the independent variable).
Column 1 is height $h$ in centimeters (the dependent variable).
The slope of the linear fit will therefore have units of cm/ns,
which can be compared directly to the speed of light
$c = 29.98\,\text{cm/ns}$.

In [ ]:
# Cell 02 - Slice data into time (ns) and height (cm) arrays

t = data[:, 0]  # time in nanoseconds
h = data[:, 1]  # height in centimeters
print(f"{t[:5] = }")
print(f"{h[:5] = }")

---
**Linear regression via the Gauss normal equations.**
The ordinary least-squares slope and intercept for $n$ data points $(x_i, y_i)$ are:

$$m = \frac{n\sum x_i y_i - \sum x_i \sum y_i}
          {n\sum x_i^2 - \left(\sum x_i\right)^2}
\qquad
b = \frac{\sum y_i - m\sum x_i}{n}$$

The slope $m$ has units cm/ns since the inputs are in those units.\
The y-intercept $b$ in this problem will be $\lt$ 3m so it is negligible.

In [ ]:
# Cell 03 - Fit a line to the height vs. time data


def fit_linear(x, y):
    """Return (slope, intercept) of the least-squares line through (x, y)."""
    n = len(x)
    m = n * np.sum(x * y) - np.sum(x) * np.sum(y)
    m /= n * np.sum(x**2) - np.sum(x) ** 2
    b = (np.sum(y) - m * np.sum(x)) / n
    return m, b


slope, yint = fit_linear(t, h)
print(f"Slope (dh/dt)  : {slope:.8f} cm/ns")
print(f"Y-intercept    : {yint:.8f} cm")

---
**Velocity and origination height.**
The speed of light is $c = 29.98\,\text{cm/ns}$, so the muon's velocity
as a fraction of $c$ is:

$$v = \frac{|\text{slope}|}{c}$$

The muon existed for $0.1743\,\text{ms}$ before impact (its lab-frame lifetime,
time-dilated from the $2.2\,\mu\text{s}$ rest-frame value).
Its origination height is:

$$h_0 = v \cdot t_{\text{lifetime}} = \frac{|\text{slope}|}{100}\times 10^9\;[\text{m/s}]
\;\times\; \frac{0.1743}{10^3}\;[\text{s}]
\;\div\; 1000\;[\text{m/km}]$$

Converting units step by step:
$|\text{slope}|\,[\text{cm/ns}] \times 10^9\,[\text{ns/s}] \div 100\,[\text{cm/m}]$
gives speed in m/s;
multiplying by $0.1743\times10^{-3}\,\text{s}$ gives distance in meters;
dividing by 1000 converts to km.

In [ ]:
# Cell 04 - Compute velocity (fraction of c) and origination height (km)

c_cm_ns = 29.98  # speed of light in cm/ns
lifetime_ms = 0.1743  # lab-frame muon lifetime in ms

v = abs(slope) / c_cm_ns  # velocity as fraction of c

# Convert slope from cm/ns to m/s, multiply by lifetime in seconds, convert to km
speed_m_s = abs(slope) * 1e9 / 100  # cm/ns -> m/s
oh = speed_m_s * (lifetime_ms / 1e3) / 1000  # m -> km

print(f"Velocity           : {v:.4f} c")
print(f"Origination height : {oh:,.2f} km")

---
**Plotting the muon trajectory.**
The blue scatter points are the raw detector measurements.
The red line is the linear regression fit.
The title shows the extracted velocity and origination height.

In [ ]:
# Cell 05 - Plot the muon trajectory in the detector

plt.figure("cosmic_rays", figsize=(10, 6))
plt.scatter(t, h, s=10, label="Detector hits")
plt.plot(t, slope * t + yint, color="red", linewidth=2, label="Best-fit line")
plt.title(
    "Secondary Cosmic Ray Muon Trajectory\n"
    f"Velocity = {v:.4f}c,  "
    f"Origination height = {oh:,.2f} km"
)
plt.xlabel("Time (ns)")
plt.ylabel("Detector Height (cm)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()